# 遙測與空間資訊之分析與應用：歷史颱風淹水足跡與道路阻斷分析

本分析使用 Google Earth Engine (GEE) 透過 Sentinel-1 合成孔徑雷達 (SAR) 影像，進行變異分析 (Change Detection)，萃取真實發生的颱風淹水範圍。接著，將淹水範圍與滿州鄉 OSM 路網進行交集，找出受災阻斷的道路。

## 1. 初始化環境與 Earth Engine

In [1]:
# 若未安裝相關套件，請取消註解並執行以下指令
# !pip install earthengine-api geemap osmnx geopandas folium matplotlib

import ee
import geemap
import osmnx as ox
import geopandas as gpd
import folium
import os
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 設定輸出目錄
output_data_dir = '../output/data'
output_map_dir = '../output/maps'
os.makedirs(output_data_dir, exist_ok=True)
os.makedirs(output_map_dir, exist_ok=True)

# 驗證並初始化 Earth Engine
try:
    ee.Initialize()
    print("Google Earth Engine 初始化成功！")
except Exception as e:
    print("尚未授權，請根據提示點擊連結進行驗證：")
    ee.Authenticate()
    ee.Initialize()
    print("Google Earth Engine 授權與初始化完成！")


Google Earth Engine 初始化成功！


## 2. 定義研究區域與下載路網

本專案位於港口溪流域，行政區隸屬「屏東縣滿州鄉」。我們將以此為範圍進行分析。

In [2]:
place_name = '滿州鄉, 屏東縣, 臺灣'
print(f"正在取得 {place_name} 的邊界與路網...")

try:
    # 取得滿州鄉邊界
    manzhou_gdf = ox.geocode_to_gdf(place_name)
    # 轉換為 Earth Engine Geometry
    geojson_bound = manzhou_gdf.geometry.iloc[0].__geo_interface__
    roi = ee.Geometry(geojson_bound)
    
    # 下載路網 (Drive)
    G = ox.graph_from_polygon(manzhou_gdf.geometry.iloc[0], network_type='drive')
    nodes, edges = ox.graph_to_gdfs(G)
    print(f"成功下載滿州鄉路網，共計 {len(edges)} 條路段。")
    
except Exception as e:
    print(f"使用地名下載失敗，改用 Bounding Box：{e}")
    # Fallback 到滿州鄉大致的 BBox
    bbox = (22.15, 21.90, 120.90, 120.75) # North, South, East, West
    roi = ee.Geometry.Rectangle([120.75, 21.90, 120.90, 22.15])
    G = ox.graph_from_bbox(bbox[0], bbox[1], bbox[2], bbox[3], network_type='drive')
    nodes, edges = ox.graph_to_gdfs(G)
    print(f"成功下載路網，共計 {len(edges)} 條路段。")

# 確保路網座標系統為 WGS84 (EPSG:4326)，以符合 GEE 標準
if edges.crs != 'epsg:4326':
    edges = edges.to_crs('epsg:4326')


正在取得 滿州鄉, 屏東縣, 臺灣 的邊界與路網...


成功下載滿州鄉路網，共計 462 條路段。


## 3. 定義 Sentinel-1 SAR 變異分析函式

使用災前與災中/災後的 VH 極化影像計算差值。由於雷達波遇到水面會發生鏡面反射，使得接收到的訊號大幅減弱，因此差值為負數（通常 < -1.25 dB）的地方即高機率為新增的積水/淹水區。

In [3]:
def extract_flood_footprint(roi, pre_start, pre_end, flood_start, flood_end, threshold=-1.25):
    '''
    從 Sentinel-1 影像中萃取淹水範圍
    '''
    # 載入 Sentinel-1 GRD 影像集合，並加入軌道方向過濾 (Orbit Pass Filter)
    collection = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterBounds(roi) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
    
    # 篩選日期
    pre_flood = collection.filterDate(pre_start, pre_end).mean().select('VH')
    during_flood = collection.filterDate(flood_start, flood_end).mean().select('VH')
    
    # 使用 Focal Median 進行平滑處理，減少 SAR 的斑點雜訊 (Speckle noise)
    smooth_radius = 50
    pre_filtered = pre_flood.focal_median(smooth_radius, 'circle', 'meters')
    during_filtered = during_flood.focal_median(smooth_radius, 'circle', 'meters')
    
    # 計算差異 (dB 值相減等同於原始值的 Log Ratio)
    difference = during_filtered.subtract(pre_filtered)
    
    # 閾值切割 (Thresholding)
    flood_mask = difference.lt(threshold)
    
    # 排除永久水體 (使用 JRC Global Surface Water 資料)
    jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select('seasonality')
    permanent_water = jrc.gte(10).unmask(0) # 存在 10 個月以上視為永久水體
    flood_mask = flood_mask.updateMask(permanent_water.eq(0))
    
    # 加入地形坡度遮罩 (Slope Mask)，排除大於 5 度的山區地形
    dem = ee.Image('USGS/SRTMGL1_003')
    slope = ee.Terrain.slope(dem)
    flat_areas = slope.lte(5)
    flood_mask = flood_mask.updateMask(flat_areas)
    
    # 清理零碎像素
    flood_mask = flood_mask.focal_mode(30, 'circle', 'meters').mask(flood_mask)
    
    return difference, flood_mask


## 4. 分析事件一：小犬颱風 (2023 年 10 月)

小犬颱風帶來強風豪雨，並造成滿州鄉港口橋斷裂與多處淹水。我們比較 9 月(災前)與 10 月初(災中)的影像。

In [ ]:
# 設定小犬颱風分析日期
koinu_pre_start = '2023-08-15'
koinu_pre_end = '2023-09-15'
koinu_flood_start = '2023-10-01'
koinu_flood_end = '2023-10-20'

# 執行淹水範圍萃取
diff_koinu, mask_koinu = extract_flood_footprint(
    roi, koinu_pre_start, koinu_pre_end, koinu_flood_start, koinu_flood_end, threshold=-1.5
)

# 將 GEE 的 Raster 轉為 Vector (Polygon)，以便後續與路網交集
# 警告：若範圍過大或雜訊過多可能會超時，滿州鄉大小通常可順利執行
print("正在將小犬颱風淹水範圍 Raster 轉換為 Vector，這可能需要幾十秒...")
koinu_vectors = mask_koinu.reduceToVectors(
    geometry=roi,
    crs=mask_koinu.projection(),
    scale=30,
    geometryType='polygon',
    eightConnected=False,
    maxPixels=1e9
)

# 透過 geemap 將 ee.FeatureCollection 轉換為本地端的 GeoDataFrame
try:
    koinu_gdf = geemap.ee_to_gdf(koinu_vectors)
    print(f"成功萃取小犬颱風淹水多邊形，共 {len(koinu_gdf)} 個區塊。")
    # 存檔備用
    koinu_gdf.to_file(os.path.join(output_data_dir, "koinu_flood_extent.geojson"), driver='GeoJSON')
except Exception as e:
    print(f"轉換失敗或查無淹水特徵: {e}")
    koinu_gdf = gpd.GeoDataFrame()


正在將小犬颱風淹水範圍 Raster 轉換為 Vector，這可能需要幾十秒...


轉換失敗或查無淹水特徵: Image.select: Band pattern 'VH' was applied to an Image with no bands. See https://developers.google.com/earth-engine/guides/debugging#no-bands


## 5. 分析事件二：丹娜絲颱風 (2019 年 7 月)

(註：使用者提到 2025/7 丹娜絲颱風，但歷史上丹娜絲為 2019/7。此處設定為 2019 年，您可自行更改日期參數)

In [ ]:
# 設定丹娜絲颱風分析日期
danas_pre_start = '2025-05-15'
danas_pre_end = '2025-06-15'
danas_flood_start = '2025-07-01'
danas_flood_end = '2025-07-20'

# 執行淹水範圍萃取
diff_danas, mask_danas = extract_flood_footprint(
    roi, danas_pre_start, danas_pre_end, danas_flood_start, danas_flood_end, threshold=-1.25
)

print("正在將丹娜絲颱風淹水範圍 Raster 轉換為 Vector...")
danas_vectors = mask_danas.reduceToVectors(
    geometry=roi,
    crs=mask_danas.projection(),
    scale=30,
    geometryType='polygon',
    eightConnected=False,
    maxPixels=1e9
)

try:
    danas_gdf = geemap.ee_to_gdf(danas_vectors)
    print(f"成功萃取丹娜絲颱風淹水多邊形，共 {len(danas_gdf)} 個區塊。")
    danas_gdf.to_file(os.path.join(output_data_dir, "danas_flood_extent.geojson"), driver='GeoJSON')
except Exception as e:
    print(f"轉換失敗或查無淹水特徵: {e}")
    danas_gdf = gpd.GeoDataFrame()


正在將丹娜絲颱風淹水範圍 Raster 轉換為 Vector...


成功萃取丹娜絲颱風淹水多邊形，共 491 個區塊。


## 6. 交通阻斷分析 (Road Blockage Intersection)

將萃取出的歷史淹水區塊，與 OSM 路段進行空間交集 (Spatial Overlay)，找出物理上被淹沒或中斷的路段。

In [6]:
def analyze_road_blockage(road_edges, flood_polygons, event_name):
    if flood_polygons.empty:
        print(f"[{event_name}] 無有效淹水圖層，無法分析道路阻斷。")
        return gpd.GeoDataFrame()
    
    # 確保座標系統一致
    flood_polygons = flood_polygons.set_crs('epsg:4326', allow_override=True)
    
    # 空間交集 (Spatial Intersection)：找出在淹水範圍內的道路
    blocked_roads = gpd.sjoin(road_edges, flood_polygons, how="inner", predicate="intersects")
    
    # 去除重複計算的路段
    blocked_roads = blocked_roads[~blocked_roads.index.duplicated(keep='first')]
    
    print(f"[{event_name}] 分析完成：總路段 {len(road_edges)} 條，受阻路段 {len(blocked_roads)} 條。")
    
    # 存檔
    blocked_roads.to_file(os.path.join(output_data_dir, f"{event_name}_blocked_roads.gpkg"), driver='GPKG')
    return blocked_roads

koinu_blocked = analyze_road_blockage(edges, koinu_gdf, "koinu")
danas_blocked = analyze_road_blockage(edges, danas_gdf, "danas")


[koinu] 無有效淹水圖層，無法分析道路阻斷。
[danas] 分析完成：總路段 462 條，受阻路段 44 條。


## 7. 視覺化展示 (小犬颱風為例)

結合原本的路網、淹水範圍與阻斷路段，輸出成互動式地圖。

In [7]:
# 以滿州鄉為中心建立 Map
m = folium.Map(location=[22.02, 120.84], zoom_start=12, tiles='CartoDB positron')

# 1. 繪製所有路網 (灰色)
folium.GeoJson(
    edges,
    name='OSM 路網',
    style_function=lambda x: {'color': '#AAAAAA', 'weight': 1, 'opacity': 0.5}
).add_to(m)

# 2. 繪製小犬颱風真實淹水範圍 (藍色半透明)
if not koinu_gdf.empty:
    folium.GeoJson(
        koinu_gdf,
        name='小犬颱風淹水範圍 (S1)',
        style_function=lambda x: {'fillColor': '#0000FF', 'color': 'none', 'fillOpacity': 0.4}
).add_to(m)

# 3. 繪製受阻路段 (紅色)
if not koinu_blocked.empty:
    folium.GeoJson(
        koinu_blocked,
        name='受阻路段 (小犬颱風)',
        style_function=lambda x: {'color': '#FF0000', 'weight': 3, 'opacity': 0.9}
).add_to(m)

# 加入圖層控制
folium.LayerControl().add_to(m)

# 存檔
map_path = os.path.join(output_map_dir, 'koinu_historical_flood_risk.html')
m.save(map_path)
print(f"互動式地圖已儲存至：{map_path}")

# 顯示地圖
m


互動式地圖已儲存至：../output/maps\koinu_historical_flood_risk.html


## 8. 視覺化展示 (丹娜絲颱風)

結合原本的路網、淹水範圍與阻斷路段，輸出成互動式地圖。

In [8]:
# 以滿州鄉為中心建立 Map
m2 = folium.Map(location=[22.02, 120.84], zoom_start=12, tiles='CartoDB positron')

# 1. 繪製所有路網 (灰色)
folium.GeoJson(
    edges,
    name='OSM 路網',
    style_function=lambda x: {'color': '#AAAAAA', 'weight': 1, 'opacity': 0.5}
).add_to(m2)

# 2. 繪製丹娜絲颱風真實淹水範圍 (藍色半透明)
if not danas_gdf.empty:
    folium.GeoJson(
        danas_gdf,
        name='丹娜絲颱風淹水範圍 (S1)',
        style_function=lambda x: {'fillColor': '#0000FF', 'color': 'none', 'fillOpacity': 0.4}
).add_to(m2)

# 3. 繪製受阻路段 (紅色)
if not danas_blocked.empty:
    folium.GeoJson(
        danas_blocked,
        name='受阻路段 (丹娜絲颱風)',
        style_function=lambda x: {'color': '#FF0000', 'weight': 3, 'opacity': 0.9}
).add_to(m2)

# 加入圖層控制
folium.LayerControl().add_to(m2)

# 存檔
map_path_danas = os.path.join(output_map_dir, 'danas_historical_flood_risk.html')
m2.save(map_path_danas)
print(f"互動式地圖已儲存至：{map_path_danas}")

# 顯示地圖
m2


互動式地圖已儲存至：../output/maps\danas_historical_flood_risk.html
